# Part 3 (optional): why OGC API?

Geonorge has offered the same data for years, as files to download and through WMS and WFS services. So what's different about an OGC API? This notebook shows it, with the API from **part 2** (run it to step 6 first) and Geonorge's own services for the same data, Kartverket's municipalities:

1. **One question, four ways:** "which municipality is this address in?"
2. **Same code, Kartverket's API**

Everything here calls Geonorge's services over the internet, so a cell can fail if one of them is down.

In [ ]:
import io
import json
import urllib.parse
import zipfile

import shapely
from pyproj import Transformer
from workshop import *

## 1. One question, four ways

The question: **which municipality is Kartverket's address in?** First, the address's coordinates, from Geonorge's address API:

In [ ]:
ADDRESS = "Kartverksveien 21, Hønefoss"   # ✏️ or any other address

hit = fetch_json("https://ws.geonorge.no/adresser/v1/sok", {"sok": ADDRESS, "treffPerSide": 1})["adresser"][0]
lon, lat = hit["representasjonspunkt"]["lon"], hit["representasjonspunkt"]["lat"]
x, y = Transformer.from_crs(4258, 25833, always_xy=True).transform(lon, lat)   # the same point in UTM 33
print(f"{hit['adressetekst']}, {hit['poststed']}: lon {lon:.5f}, lat {lat:.5f} (UTM 33: {x:.0f}, {y:.0f})")

### a) Download a file from Geonorge

The way most people get Kartverket's data today. Geonorge's download API takes an order (dataset, area, format and projection) and hands back a link to a zip file. You can't order "the municipality at this point", only whole areas, so you get every municipality in Norway and search the file yourself.

In [ ]:
order = {"orderLines": [{
    "metadataUuid": "041f1e6e-bdbc-4091-b48f-8a5990f3cc5b",   # the dataset "Kommuner" in Geonorge
    "areas": [{"code": "0000", "type": "landsdekkende", "name": "Hele landet"}],
    "formats": [{"name": "GeoJSON "}],
    "projections": [{"code": "25833"}],
}]}
receipt = json.loads(fetch("https://nedlasting.geonorge.no/api/order", "File download",
                           data=json.dumps(order).encode(), content_type="application/json"))
zipped = fetch(receipt["files"][0]["downloadUrl"], "File download")

# Unzip it and search the municipalities in it, on your own machine.
# The file also holds every piece of boundary line ("Grense"), so keep only the "Kommune" polygons.
with zipfile.ZipFile(io.BytesIO(zipped)) as z:
    data = json.loads(z.read(z.namelist()[0]))
kommuner = [f for f in data["features"] if f["properties"].get("objtype") == "Kommune"]
point = shapely.Point(x, y)
found = [f["properties"] for f in kommuner if shapely.geometry.shape(f["geometry"]).intersects(point)]
print(f"\nThe file has {len(data['features'])} features: {len(kommuner)} municipalities "
      f"and {len(data['features']) - len(kommuner)} pieces of boundary line")
print(f"Searched the {len(kommuner)} municipalities:", [(f.get("kommunenavn"), f.get("kommunenummer")) for f in found])
extracted = kommuner[0]["properties"]["datauttaksdato"]
print(f"The file was extracted from the database on {extracted[:4]}-{extracted[4:6]}-{extracted[6:8]}")

### b) WMS

A WMS draws a **picture**. It can tell you what's at a point, but you ask about a *pixel*: "in a 500 × 500 picture of this area, what's at pixel (250, 250)?" The answer is text, meant for a popup.

In [ ]:
from IPython.display import Image, display

WMS = "https://wms.geonorge.no/skwms1/wms.adm_enheter2"
area = {"service": "WMS", "version": "1.3.0", "layers": "kommuner", "styles": "", "crs": "EPSG:25833",
        "bbox": f"{x - 30000},{y - 30000},{x + 30000},{y + 30000}", "width": 500, "height": 500}

picture = fetch(WMS + "?" + urllib.parse.urlencode({**area, "request": "GetMap", "format": "image/png"}), "WMS")
display(Image(picture))

info = fetch(WMS + "?" + urllib.parse.urlencode({**area, "request": "GetFeatureInfo", "query_layers": "kommuner",
                                                 "info_format": "text/plain", "i": 250, "j": 250}), "WMS")
print(info.decode()[:600])

### c) WFS

A WFS sends **data**, as GML (XML). A spatial question is written in XML too, and sent as a POST:

In [ ]:
request = f"""<wfs:GetFeature service="WFS" version="2.0.0"
    xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:fes="http://www.opengis.net/fes/2.0"
    xmlns:gml="http://www.opengis.net/gml/3.2"
    xmlns:app="https://skjema.geonorge.no/SOSI/produktspesifikasjon/AdmEnheter/20240101">
  <wfs:Query typeNames="app:Kommune">
    <fes:Filter>
      <fes:Intersects>
        <fes:ValueReference>app:område</fes:ValueReference>
        <gml:Point gml:id="p1" srsName="urn:ogc:def:crs:EPSG::4258"><gml:pos>{lat} {lon}</gml:pos></gml:Point>
      </fes:Intersects>
    </fes:Filter>
  </wfs:Query>
</wfs:GetFeature>"""

gml = fetch("https://wfs.geonorge.no/skwms1/wfs.administrative_enheter", "WFS",
            data=request.encode(), content_type="application/xml").decode()
print(gml[:800], "…")

Notice `<gml:pos>{lat} {lon}</gml:pos>`: EPSG:4258 puts latitude first, so the coordinates go in the "wrong" order. Getting that wrong is a classic WFS mistake. (A WFS can leave out the geometry too, with more XML: `wfs:PropertyName`.)

### d) Your OGC API

One URL, which works in a browser, QGIS, Excel or a script alike:

In [ ]:
url = "http://localhost:3000/collections/kommuner/items?" + urllib.parse.urlencode({
    "filter": f"S_INTERSECTS(geometry, POINT({lon} {lat}))",
    "skipGeometry": "true",
    "f": "json",
})
print(urllib.parse.unquote_plus(url), "\n")
answer = json.loads(fetch(url, "OGC API"))
print([(f["properties"]["navn"], f["id"]) for f in answer["features"]])

### What did each answer cost?

In [ ]:
compare()

All four found the same municipality, but:

- **The file** answered a question about one municipality by sending all of them, plus every piece of their boundary lines, and it's a snapshot: look at the date it was extracted. Every update means a new download, while your API reads the database live (step 10 of part 2).
- **The WMS** is a picture. Fine for a background map, but you can't calculate with it or change how it looks, the way you styled the vector tiles in step 9 of part 2.
- **The WFS** has the data, but in XML, with requests written in XML and a schema of its own to learn.
- **The OGC API** answered with one readable URL and a few hundred bytes of JSON, straight from the database.

The times change from run to run with the network and how busy each server is. Your OGC API runs in the Codespace, so its time skips the trip over the internet the other three make; the file's time doesn't count unzipping and searching it on your machine.

## 2. Same code, Kartverket's API

Because OGC API is a standard, code written for your API works on any OGC API. Below is the question from section 1d again, with the same code, sent to Kartverket's own demo service, which runs the same software as yours. Only the address is different. It's a test environment, so it may now and then be down.

In [ ]:
KARTVERKET = "https://datadeling-ogcapi.atgcp1-dev.kartverket.cloud"

for c in fetch_json(f"{KARTVERKET}/collections", {"f": "json"})["collections"]:
    print(c["id"], "-", c["title"])

# The question from 1d: which municipality is the address in?
url = f"{KARTVERKET}/collections/kommuner/items?" + urllib.parse.urlencode({
    "filter": f"S_INTERSECTS(geometry, POINT({lon} {lat}))",
    "skipGeometry": "true",
    "f": "json",
})
print("\n" + urllib.parse.unquote_plus(url), "\n")
answer = json.loads(fetch(url, "Kartverket's OGC API"))
print([(f["properties"]["navn"], f["id"]) for f in answer["features"]])